# EcoSignal chlorosis retraining v2

This Colab notebook rebuilds chlorosis assessment around the retrained **320×320 Keras lichen-coverage model**. It does **not** require pixel-level chlorosis masks: it uses the accepted lichen mask to isolate lichen, then learns from one independent image-level chlorosis reference score per photograph.

The notebook deliberately separates: (1) coverage-mask generation and review, (2) color correction and feature extraction, (3) leakage-safe validation by tree/site, and (4) production export. It will not export a production chlorosis model unless minimum scientific validation gates pass.

> Chlorosis is a visual stress indicator. It does not identify a pollutant or establish causation. AQI should remain a separate, time-stamped evidence source until an AQI-aware risk engine is independently validated.

## 1. Required inputs

Put the retrained coverage model and field images in Google Drive. Each image needs an independent reference score from `0.0` (no visible chlorosis) to `1.0` (severe visible chlorosis), plus `tree_id` and `site_id`. A blinded expert consensus, standardized color-chart/Munsell protocol, or pigment measurement is preferable. Do not create the score from the same RGB features this notebook evaluates.

For production-quality color analysis, each row must also document either standardized diffuse lighting or a gray reference. You only approve/reject the **coverage mask**; you do not draw a chlorosis mask.

In [ ]:
%pip install -q tensorflow==2.18.0 opencv-python-headless scikit-image scikit-learn pandas scipy joblib matplotlib seaborn tqdm
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# EDIT ONLY THESE PATHS IF YOUR DRIVE LAYOUT IS DIFFERENT.
PROJECT_DIR = Path('/content/drive/MyDrive/EcoSignal/chlorosis_retraining_v2')
IMAGE_DIR = PROJECT_DIR / 'images'
MANIFEST_PATH = PROJECT_DIR / 'chlorosis_manifest.csv'
COVERAGE_MODEL_PATH = Path('/content/drive/MyDrive/EcoSignal/ecosignal-backend/models/lichen_coverage_model.keras')
OUTPUT_DIR = PROJECT_DIR / 'outputs'
ARTIFACT_DIR = OUTPUT_DIR / 'coverage_review'

COVERAGE_INPUT_SIZE = 320
COVERAGE_THRESHOLD = 0.50
MIN_COVERAGE_PERCENT = 10.0  # current model is best validated at high coverage
MIN_LICHEN_PIXELS = 1000
RANDOM_SEED = 42
N_GROUP_FOLDS = 5
REFERENCE_SCALE = '0=no visible chlorosis; 1=severe visible chlorosis'

for folder in (PROJECT_DIR, IMAGE_DIR, OUTPUT_DIR, ARTIFACT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
assert COVERAGE_MODEL_PATH.is_file(), f'Coverage model not found: {COVERAGE_MODEL_PATH}'
print('Project:', PROJECT_DIR)
print('Coverage model:', COVERAGE_MODEL_PATH)

## 2. Create or validate the manifest

On the first run, this cell creates `chlorosis_manifest.csv`. Open it in Google Sheets or Excel and fill the required columns. Keep `mask_approved` false until you inspect the generated overlay later in the notebook.

Required columns: `image_path`, `reference_score`, `tree_id`, `site_id`, `standardized_lighting`, and `mask_approved`. Optional gray-reference RGB values improve color correction; optional metadata is retained for subgroup checks.

In [ ]:
import pandas as pd

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.webp'}
REQUIRED_COLUMNS = [
    'image_path', 'reference_score', 'tree_id', 'site_id',
    'standardized_lighting', 'mask_approved',
    'gray_r', 'gray_g', 'gray_b', 'device', 'growth_form', 'color_type'
]

if not MANIFEST_PATH.exists():
    images = sorted(p for p in IMAGE_DIR.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
    template = pd.DataFrame({
        'image_path': [str(p) for p in images],
        'reference_score': [pd.NA] * len(images),
        'tree_id': [pd.NA] * len(images),
        'site_id': [pd.NA] * len(images),
        'standardized_lighting': [False] * len(images),
        'mask_approved': [False] * len(images),
        'gray_r': [pd.NA] * len(images), 'gray_g': [pd.NA] * len(images), 'gray_b': [pd.NA] * len(images),
        'device': [pd.NA] * len(images), 'growth_form': [pd.NA] * len(images), 'color_type': [pd.NA] * len(images),
    })
    template.to_csv(MANIFEST_PATH, index=False)
    print(f'Created {MANIFEST_PATH} with {len(images)} images.')
    print('Fill reference_score/tree_id/site_id/lighting metadata, then rerun this cell.')

manifest = pd.read_csv(MANIFEST_PATH)
missing = sorted(set(REQUIRED_COLUMNS) - set(manifest.columns))
assert not missing, f'Manifest is missing columns: {missing}'
manifest['image_path'] = manifest['image_path'].astype(str)
manifest['reference_score'] = pd.to_numeric(manifest['reference_score'], errors='coerce')
for column in ['standardized_lighting', 'mask_approved']:
    manifest[column] = manifest[column].astype(str).str.lower().isin(['true', '1', 'yes', 'y'])
bad_paths = [p for p in manifest.image_path if not Path(p).is_file()]
assert not bad_paths, f'Missing image files (first 5): {bad_paths[:5]}'
if manifest.reference_score.notna().any():
    assert manifest.reference_score.dropna().between(0, 1).all(), 'reference_score must be between 0 and 1'
display(manifest.head())
print(f'Rows: {len(manifest)} | labeled: {manifest.reference_score.notna().sum()} | masks approved: {manifest.mask_approved.sum()}')

## 3. Load the exact updated coverage model and define mask QA

Preprocessing matches EcoSignal backend v1.1: RGB, direct resize to 320×320, float32 divided by 255, and a 0.50 sigmoid threshold. The predicted mask is resized back to original resolution before chlorosis features are extracted.

In [ ]:
import hashlib, json, math, warnings
import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from PIL import Image, ImageOps
from scipy.stats import spearmanr
from skimage.color import rgb2lab, rgb2hsv
from tqdm.auto import tqdm

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

coverage_model = tf.keras.models.load_model(COVERAGE_MODEL_PATH, compile=False)
test_shape = tuple(coverage_model.output_shape)
assert test_shape[-3:] in [(320, 320, 1), (320, 320)], f'Unexpected model output: {test_shape}'
COVERAGE_MODEL_SHA256 = sha256(COVERAGE_MODEL_PATH)
print('Coverage model SHA-256:', COVERAGE_MODEL_SHA256)
print('Input:', coverage_model.input_shape, '| Output:', coverage_model.output_shape)

def predict_coverage(image_path):
    image = ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
    rgb = np.asarray(image)
    height, width = rgb.shape[:2]
    resized = np.asarray(image.resize((COVERAGE_INPUT_SIZE, COVERAGE_INPUT_SIZE), Image.Resampling.BILINEAR), dtype=np.float32) / 255.0
    pred = np.asarray(coverage_model.predict(resized[None, ...], verbose=0)).squeeze().astype(np.float32)
    assert pred.shape == (COVERAGE_INPUT_SIZE, COVERAGE_INPUT_SIZE), pred.shape
    probability = np.asarray(Image.fromarray(pred, mode='F').resize((width, height), Image.Resampling.BILINEAR))
    mask = probability >= COVERAGE_THRESHOLD
    inside = probability[mask]
    outside = probability[~mask]
    mean_inside = float(inside.mean()) if inside.size else 0.0
    p90_outside = float(np.percentile(outside, 90)) if outside.size else 1.0
    separation = float(np.clip(mean_inside - p90_outside, 0, 1))
    coverage_percent = float(100 * mask.mean())
    confidence = float(np.clip(0.55 * mean_inside + 0.30 * separation + 0.15 * min(coverage_percent / 20, 1), 0, 1))
    if coverage_percent < 10:
        confidence *= 0.80
    return rgb, probability, mask, {'coverage_percent': coverage_percent, 'coverage_confidence': confidence, 'mask_pixels': int(mask.sum())}

def save_coverage_artifacts(image_path, rgb, probability, mask):
    stem = hashlib.sha1(str(image_path).encode()).hexdigest()[:12]
    mask_path = ARTIFACT_DIR / f'{stem}_mask.png'
    overlay_path = ARTIFACT_DIR / f'{stem}_overlay.jpg'
    probability_path = ARTIFACT_DIR / f'{stem}_probability.png'
    Image.fromarray((mask * 255).astype(np.uint8)).save(mask_path)
    color = rgb.copy(); color[mask] = [255, 70, 70]
    Image.fromarray(np.clip(0.68 * rgb + 0.32 * color, 0, 255).astype(np.uint8)).save(overlay_path, quality=92)
    Image.fromarray(np.clip(probability * 255, 0, 255).astype(np.uint8)).save(probability_path)
    return str(mask_path), str(overlay_path), str(probability_path)

## 4. Generate masks and review overlays

Run this once, inspect the overlays in `outputs/coverage_review`, and change `mask_approved` to true only for masks that correctly isolate lichen. Images below 10% predicted coverage are retained in the audit but excluded from the current production training scope. After editing the CSV, rerun the manifest cell and continue.

In [ ]:
mask_records = []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='Coverage masks'):
    try:
        rgb, probability, mask, qa = predict_coverage(row.image_path)
        mask_path, overlay_path, probability_path = save_coverage_artifacts(row.image_path, rgb, probability, mask)
        mask_records.append({'image_path': row.image_path, **qa, 'coverage_mask_path': mask_path, 'coverage_overlay_path': overlay_path, 'coverage_probability_path': probability_path, 'coverage_error': ''})
    except Exception as exc:
        mask_records.append({'image_path': row.image_path, 'coverage_error': f'{type(exc).__name__}: {exc}'})

mask_audit = pd.DataFrame(mask_records)
mask_audit.to_csv(OUTPUT_DIR / 'coverage_mask_audit.csv', index=False)
audit_manifest = manifest.drop(columns=[c for c in mask_audit.columns if c != 'image_path' and c in manifest.columns]).merge(mask_audit, on='image_path', how='left')
audit_manifest.to_csv(OUTPUT_DIR / 'chlorosis_manifest_with_mask_audit.csv', index=False)
display(audit_manifest[['image_path', 'coverage_percent', 'coverage_confidence', 'mask_approved', 'coverage_overlay_path', 'coverage_error']].head(20))
print('Review overlays here:', ARTIFACT_DIR)
print('Then update mask_approved in:', MANIFEST_PATH)

valid_previews = audit_manifest[audit_manifest.coverage_error.fillna('').eq('')].head(12)
if len(valid_previews):
    cols = 3; rows = math.ceil(len(valid_previews) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows)); axes = np.atleast_1d(axes).ravel()
    for ax, (_, item) in zip(axes, valid_previews.iterrows()):
        ax.imshow(Image.open(item.coverage_overlay_path)); ax.axis('off')
        ax.set_title(f"{Path(item.image_path).name}\ncoverage={item.coverage_percent:.1f}% conf={item.coverage_confidence:.2f}")
    for ax in axes[len(valid_previews):]: ax.axis('off')
    plt.tight_layout(); plt.show()

## 5. Color correction and lichen-only feature extraction

The mask is eroded slightly before analysis to reduce bark contamination at boundaries. Gray-reference correction is applied when RGB values are supplied. Features summarize color distributions rather than classifying individual pixels as chlorotic, so no chlorosis masks are required.

In [ ]:
def correct_gray(rgb, gray_values):
    arr = rgb.astype(np.float32) / 255.0
    if gray_values is None:
        return arr, False
    gray = np.asarray(gray_values, dtype=np.float32) / 255.0
    if gray.shape != (3,) or np.any(~np.isfinite(gray)) or np.any(gray <= 0):
        raise ValueError('gray_r/gray_g/gray_b must be three positive RGB values')
    target = float(gray.mean())
    return np.clip(arr * (target / gray)[None, None, :], 0, 1), True

def summarize(values, prefix):
    values = np.asarray(values, dtype=np.float32)
    return {
        f'{prefix}_mean': float(np.mean(values)), f'{prefix}_std': float(np.std(values)),
        f'{prefix}_p10': float(np.percentile(values, 10)), f'{prefix}_p25': float(np.percentile(values, 25)),
        f'{prefix}_median': float(np.median(values)), f'{prefix}_p75': float(np.percentile(values, 75)),
        f'{prefix}_p90': float(np.percentile(values, 90)),
    }

def extract_color_features(rgb_uint8, mask, gray_values=None):
    kernel = np.ones((3, 3), np.uint8)
    eroded = cv2.erode(mask.astype(np.uint8), kernel, iterations=1).astype(bool)
    if eroded.sum() < MIN_LICHEN_PIXELS:
        eroded = mask.astype(bool)
    if eroded.sum() < MIN_LICHEN_PIXELS:
        raise ValueError(f'Only {eroded.sum()} usable lichen pixels; need {MIN_LICHEN_PIXELS}')
    rgb, gray_applied = correct_gray(rgb_uint8, gray_values)
    lab = rgb2lab(rgb)
    hsv = rgb2hsv(rgb)
    px = rgb[eroded]; lab_px = lab[eroded]; hsv_px = hsv[eroded]
    total = np.maximum(px.sum(axis=1), 1e-6)
    r, g, b = px[:, 0], px[:, 1], px[:, 2]
    signals = {
        'r': r, 'g': g, 'b': b,
        'r_chroma': r / total, 'g_chroma': g / total, 'b_chroma': b / total,
        'green_red_nd': (g - r) / np.maximum(g + r, 1e-6),
        'yellow_blue_nd': (0.5 * (r + g) - b) / np.maximum(0.5 * (r + g) + b, 1e-6),
        'excess_green': 2 * g - r - b,
        'lab_l': lab_px[:, 0] / 100.0, 'lab_a': lab_px[:, 1] / 128.0, 'lab_b': lab_px[:, 2] / 128.0,
        'hsv_h': hsv_px[:, 0], 'hsv_s': hsv_px[:, 1], 'hsv_v': hsv_px[:, 2],
    }
    features = {}
    for name, values in signals.items(): features.update(summarize(values, name))
    features.update({'usable_lichen_pixels': int(eroded.sum()), 'gray_reference_applied': bool(gray_applied)})
    return features, eroded

# Reload the edited approval column and join the previously generated mask audit.
manifest = pd.read_csv(MANIFEST_PATH)
for column in ['standardized_lighting', 'mask_approved']:
    manifest[column] = manifest[column].astype(str).str.lower().isin(['true', '1', 'yes', 'y'])
manifest['reference_score'] = pd.to_numeric(manifest['reference_score'], errors='coerce')
mask_audit = pd.read_csv(OUTPUT_DIR / 'coverage_mask_audit.csv')
data = manifest.merge(mask_audit, on='image_path', how='left')

feature_records, extraction_errors = [], []
for row in tqdm(data.itertuples(index=False), total=len(data), desc='Color features'):
    try:
        rgb = np.asarray(ImageOps.exif_transpose(Image.open(row.image_path)).convert('RGB'))
        mask = np.asarray(Image.open(row.coverage_mask_path).convert('L')) > 127
        gray_cols = [getattr(row, c) for c in ('gray_r', 'gray_g', 'gray_b')]
        gray = None if any(pd.isna(v) for v in gray_cols) else [float(v) for v in gray_cols]
        feats, eroded = extract_color_features(rgb, mask, gray)
        feature_records.append({'image_path': row.image_path, **feats})
    except Exception as exc:
        extraction_errors.append({'image_path': row.image_path, 'error': f'{type(exc).__name__}: {exc}'})

features_df = pd.DataFrame(feature_records)
model_table = data.merge(features_df, on='image_path', how='inner')
model_table.to_csv(OUTPUT_DIR / 'chlorosis_feature_table.csv', index=False)
pd.DataFrame(extraction_errors).to_csv(OUTPUT_DIR / 'feature_extraction_errors.csv', index=False)
print('Extracted:', len(features_df), '| Errors:', len(extraction_errors))

## 6. Apply scientific eligibility gates

Training uses only labeled images with approved masks, sufficient predicted coverage, and documented standardized lighting or a gray reference. It groups all images from the same site/tree together so near-duplicate samples cannot leak across folds.

In [ ]:
has_gray = model_table[['gray_r', 'gray_g', 'gray_b']].notna().all(axis=1)
eligible = model_table[
    model_table.reference_score.notna()
    & model_table.reference_score.between(0, 1)
    & model_table.mask_approved
    & (model_table.coverage_percent >= MIN_COVERAGE_PERCENT)
    & (model_table.standardized_lighting | has_gray)
    & model_table.coverage_error.fillna('').eq('')
].copy()
eligible['group_id'] = eligible.site_id.astype(str) + '::' + eligible.tree_id.astype(str)

reasons = pd.DataFrame({
    'total_rows': [len(model_table)],
    'eligible_rows': [len(eligible)],
    'unlabeled': [model_table.reference_score.isna().sum()],
    'mask_not_approved': [(~model_table.mask_approved).sum()],
    'below_coverage_scope': [(model_table.coverage_percent < MIN_COVERAGE_PERCENT).sum()],
    'lighting_not_documented': [(~(model_table.standardized_lighting | has_gray)).sum()],
    'unique_trees': [eligible.group_id.nunique()],
})
display(reasons)
display(eligible.reference_score.describe())
assert len(eligible) >= 30, 'Need at least 30 eligible labeled images to run preliminary validation (60+ strongly recommended).'
assert eligible.group_id.nunique() >= 5, 'Need at least 5 distinct site/tree groups for grouped validation.'
assert eligible.reference_score.nunique() >= 4, 'Reference scores need at least four distinct severity levels.'

## 7. Leakage-safe model comparison

Two fixed candidates are compared with out-of-fold predictions: a transparent regularized linear model and a nonlinear random forest. Model selection is based on grouped cross-validation MAE, with Spearman rank correlation reported separately.

In [ ]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

NON_FEATURE_COLUMNS = set(REQUIRED_COLUMNS + [
    'coverage_percent', 'coverage_confidence', 'mask_pixels', 'coverage_mask_path',
    'coverage_overlay_path', 'coverage_probability_path', 'coverage_error',
    'group_id', 'usable_lichen_pixels', 'gray_reference_applied'
])
FEATURE_COLUMNS = [c for c in eligible.columns if c not in NON_FEATURE_COLUMNS and pd.api.types.is_numeric_dtype(eligible[c])]
assert FEATURE_COLUMNS, 'No numeric color features found.'
X = eligible[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
y = eligible.reference_score.astype(float).to_numpy()
groups = eligible.group_id.to_numpy()
folds = min(N_GROUP_FOLDS, len(np.unique(groups)))
splitter = GroupKFold(n_splits=folds)

models = {
    'ridge_transparent': Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()), ('model', Ridge(alpha=10.0))]),
    'random_forest': Pipeline([('impute', SimpleImputer(strategy='median')), ('model', RandomForestRegressor(n_estimators=500, min_samples_leaf=3, max_features=0.6, random_state=RANDOM_SEED, n_jobs=-1))]),
}

oof_predictions, metric_rows = {}, []
for model_name, estimator in models.items():
    pred = np.full(len(eligible), np.nan, dtype=float)
    fold_ids = np.full(len(eligible), -1, dtype=int)
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups)):
        fitted = clone(estimator).fit(X.iloc[train_idx], y[train_idx])
        pred[test_idx] = np.clip(fitted.predict(X.iloc[test_idx]), 0, 1)
        fold_ids[test_idx] = fold
    rho = float(spearmanr(y, pred).statistic)
    calibration_bias = float(abs(np.mean(pred) - np.mean(y)))
    calibration_slope = float(np.polyfit(pred, y, 1)[0]) if np.std(pred) > 1e-8 else float('nan')
    metric_rows.append({'model': model_name, 'mae': mean_absolute_error(y, pred), 'rmse': mean_squared_error(y, pred) ** 0.5, 'spearman_rho': rho, 'calibration_bias': calibration_bias, 'calibration_slope': calibration_slope})
    oof_predictions[model_name] = pred

metrics = pd.DataFrame(metric_rows).sort_values(['mae', 'spearman_rho'], ascending=[True, False]).reset_index(drop=True)
display(metrics)
BEST_MODEL_NAME = metrics.iloc[0].model
BEST_OOF = oof_predictions[BEST_MODEL_NAME]
print('Selected:', BEST_MODEL_NAME)

oof = eligible[['image_path', 'tree_id', 'site_id', 'device', 'growth_form', 'color_type', 'coverage_percent', 'reference_score']].copy()
for name, values in oof_predictions.items(): oof[f'pred_{name}'] = values
oof.to_csv(OUTPUT_DIR / 'chlorosis_oof_predictions.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'chlorosis_model_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, BEST_OOF, alpha=.7); ax.plot([0, 1], [0, 1], '--', color='black')
ax.set(xlabel='Independent reference score', ylabel='Out-of-fold prediction', title=f'{BEST_MODEL_NAME}: grouped validation')
plt.show()

## 8. Subgroup audit and production gate

The gate is intentionally conservative. A candidate production export needs at least 100 eligible images from 20 tree/site groups, representation at both ends of the severity scale, MAE ≤ 0.15, Spearman ρ ≥ 0.50, and mean calibration bias ≤ 0.05. Passing does not prove pollutant causation; it only supports estimating the chosen visual chlorosis reference under the validated capture conditions.

In [ ]:
eligible = eligible.copy()
eligible['oof_prediction'] = BEST_OOF
eligible['absolute_error'] = np.abs(eligible.reference_score - eligible.oof_prediction)
eligible['coverage_band'] = pd.cut(eligible.coverage_percent, bins=[0, 10, 25, 50, 100], labels=['<10', '10-25', '25-50', '50-100'], include_lowest=True)

subgroup_tables = {}
for column in ['coverage_band', 'device', 'growth_form', 'color_type', 'standardized_lighting']:
    table = eligible.groupby(column, dropna=False, observed=True).agg(n=('absolute_error', 'size'), mae=('absolute_error', 'mean')).reset_index()
    subgroup_tables[column] = table
    print('\n', column); display(table)

best_metrics = metrics.iloc[0].to_dict()
gate_checks = {
    'eligible_images_at_least_100': len(eligible) >= 100,
    'independent_groups_at_least_20': eligible.group_id.nunique() >= 20,
    'at_least_10_low_severity_examples': int((eligible.reference_score <= 0.25).sum()) >= 10,
    'at_least_10_high_severity_examples': int((eligible.reference_score >= 0.75).sum()) >= 10,
    'mae_at_most_0_15': float(best_metrics['mae']) <= 0.15,
    'spearman_at_least_0_50': float(best_metrics['spearman_rho']) >= 0.50,
    'calibration_bias_at_most_0_05': float(best_metrics['calibration_bias']) <= 0.05,
    'all_masks_approved': bool(eligible.mask_approved.all()),
    'all_lighting_documented': bool((eligible.standardized_lighting | eligible[['gray_r', 'gray_g', 'gray_b']].notna().all(axis=1)).all()),
}
PRODUCTION_GATE_PASSED = all(gate_checks.values())
print(json.dumps(gate_checks, indent=2))
print('PRODUCTION GATE:', 'PASS' if PRODUCTION_GATE_PASSED else 'FAIL — keep chlorosis disabled in the app')

## 9. Fit and export the validated bundle

If the gate fails, the notebook still writes a validation report and exploratory outputs, but it will not write a production model. Improve data quantity, label independence, capture standardization, or subgroup performance and rerun.

In [ ]:
from datetime import datetime, timezone

report = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'status': 'production_gate_passed' if PRODUCTION_GATE_PASSED else 'exploratory_only_gate_failed',
    'reference_scale': REFERENCE_SCALE,
    'coverage_model_sha256': COVERAGE_MODEL_SHA256,
    'coverage_contract': {'input_size': 320, 'rgb_divisor': 255.0, 'threshold': COVERAGE_THRESHOLD, 'minimum_validated_coverage_percent': MIN_COVERAGE_PERCENT},
    'selected_model': BEST_MODEL_NAME,
    'eligible_images': int(len(eligible)),
    'independent_groups': int(eligible.group_id.nunique()),
    'feature_columns': FEATURE_COLUMNS,
    'metrics': [{k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in row.items()} for row in metrics.to_dict('records')],
    'gate_checks': {k: bool(v) for k, v in gate_checks.items()},
    'limitations': [
        'Visual chlorosis score does not identify a pollutant or establish causation.',
        'Model is valid only under documented standardized lighting or gray-reference correction.',
        f'Current scope excludes predicted lichen coverage below {MIN_COVERAGE_PERCENT}%.',
        'Performance must be rechecked for new devices, sites, growth forms, and lichen color types.',
    ],
}
with open(OUTPUT_DIR / 'chlorosis_validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

if PRODUCTION_GATE_PASSED:
    final_model = clone(models[BEST_MODEL_NAME]).fit(X, y)
    bundle = {
        'model': final_model, 'feature_columns': FEATURE_COLUMNS,
        'version': 'chlorosis_coverage320_groupvalidated_v2',
        'coverage_model_sha256': COVERAGE_MODEL_SHA256,
        'coverage_threshold': COVERAGE_THRESHOLD,
        'minimum_coverage_percent': MIN_COVERAGE_PERCENT,
        'reference_scale': REFERENCE_SCALE,
        'validation_metrics': best_metrics,
    }
    model_out = OUTPUT_DIR / 'chlorosis_model_bundle.joblib'
    joblib.dump(bundle, model_out)
    report['chlorosis_bundle_sha256'] = sha256(model_out)
    with open(OUTPUT_DIR / 'chlorosis_validation_report.json', 'w') as f: json.dump(report, f, indent=2)
    print('Production bundle:', model_out)
    print('Bundle SHA-256:', report['chlorosis_bundle_sha256'])
else:
    print('No production model exported. Read chlorosis_validation_report.json and address failed gates.')

## 10. What to send back for backend integration

After a passing run, provide these files:

- `outputs/chlorosis_model_bundle.joblib`
- `outputs/chlorosis_validation_report.json`
- `outputs/chlorosis_model_comparison.csv`
- `outputs/chlorosis_oof_predictions.csv`
- `outputs/chlorosis_feature_table.csv`
- the completed `chlorosis_manifest.csv` (image paths may be redacted, but tree/site grouping and metadata must remain auditable)

Do not enable chlorosis in EcoSignal if the production gate says FAIL. The existing app should continue returning `revalidation required` until the bundle and report pass acceptance checks.